# Heart Disease Prediction & Classification ML Lab
### End-to-End Supervised Machine Learning Pipeline & Deep Learning Benchmark

**Dataset:** Cleveland Heart Disease Dataset (UCI Machine Learning Repository)
**Objective:** Predict the presence (`1`) or absence (`0`) of cardiovascular disease using clinical and physiological markers.
**Pipeline Highlights:**
- Statistical Profiling & Exploratory Data Analysis (Chi-Square independence, Pearson correlation)
- Preprocessing & Feature Scaling (`StandardScaler`, Outlier filtering)
- Baseline to Advanced Model Benchmarking: `DummyClassifier`, `LogisticRegression`, `RandomForestClassifier`, `ExtraTreesClassifier`, `HistGradientBoostingClassifier`, `SVC (RBF)`
- Bayesian Hyperparameter Optimization with **Optuna**
- Deep Learning Multi-Layer Perceptron (MLP) built with **TensorFlow / Keras**
- Diagnostic Model Evaluation: ROC-AUC curves, Precision-Recall curves, Confusion Matrices, and Classification Reports

In [ ]:
# 1. Environment & Library Setup
!pip install -q optuna tensorflow ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
import optuna

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

In [ ]:
# 2. Data Loading & Feature Verification
try:
    url = "https://raw.githubusercontent.com/datasets/heart-disease/master/data/heart-disease.csv"
    df = pd.read_csv(url)
    print(f"Loaded dataset from remote repository: {df.shape}")
except Exception as e:
    print("Loading synthetic Cleveland clinical dataset for offline demonstration:", e)
    np.random.seed(42)
    n = 303
    df = pd.DataFrame({
        "age": np.random.randint(29, 77, n),
        "sex": np.random.choice([0, 1], n, p=[0.32, 0.68]),
        "cp": np.random.choice([0, 1, 2, 3], n),
        "trestbps": np.random.randint(94, 200, n),
        "chol": np.random.randint(126, 400, n),
        "fbs": np.random.choice([0, 1], n, p=[0.85, 0.15]),
        "restecg": np.random.choice([0, 1, 2], n),
        "thalach": np.random.randint(71, 202, n),
        "exang": np.random.choice([0, 1], n, p=[0.67, 0.33]),
        "oldpeak": np.round(np.random.exponential(1.0, n), 1),
        "slope": np.random.choice([0, 1, 2], n),
        "ca": np.random.choice([0, 1, 2, 3], n, p=[0.6, 0.2, 0.13, 0.07]),
        "thal": np.random.choice([1, 2, 3], n),
        "target": np.random.choice([0, 1], n, p=[0.46, 0.54])
    })

df.columns = [c.strip().lower() for c in df.columns]
target_col = "target" if "target" in df.columns else df.columns[-1]
df[target_col] = (df[target_col] > 0).astype(int)
print("Summary statistics:\n", df.describe().T[["mean", "std", "min", "50%", "max"]])

In [ ]:
# 3. Exploratory Data Analysis & Statistical Association
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x=target_col, palette=["#2ecc71", "#e74c3c"], ax=axes[0])
axes[0].set_title(f"Target Distribution (Heart Disease: {target_col})")
axes[0].set_xticklabels(["Absence (0)", "Presence (1)"])

sns.boxplot(data=df, x=target_col, y="thalach", palette=["#2ecc71", "#e74c3c"], ax=axes[1])
axes[1].set_title("Maximum Heart Rate (thalach) vs Disease")
axes[1].set_xticklabels(["Absence", "Presence"])

sns.boxplot(data=df, x=target_col, y="oldpeak", palette=["#2ecc71", "#e74c3c"], ax=axes[2])
axes[2].set_title("ST Depression (oldpeak) vs Disease")
axes[2].set_xticklabels(["Absence", "Presence"])
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Matrix of Clinical Markers")
plt.show()

In [ ]:
# 4. Train-Test Split & Standardization
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")

In [ ]:
# 5. Model Suite Benchmarking
models = {
    "Dummy (Majority)": (DummyClassifier(strategy="most_frequent"), X_train),
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), X_train_scaled),
    "Random Forest": (RandomForestClassifier(n_estimators=100, random_state=42), X_train),
    "Extra Trees": (ExtraTreesClassifier(n_estimators=100, random_state=42), X_train),
    "HistGradientBoosting": (HistGradientBoostingClassifier(random_state=42), X_train),
    "Support Vector Machine (RBF)": (SVC(probability=True, random_state=42), X_train_scaled)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
records = []

for name, (clf, data_train) in models.items():
    scores = cross_val_score(clf, data_train, y_train, cv=cv, scoring="roc_auc")
    clf.fit(data_train, y_train)
    data_test = X_test_scaled if "scaled" in str(data_train) or name in ["Logistic Regression", "Support Vector Machine (RBF)"] else X_test
    y_proba = clf.predict_proba(data_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)
    records.append({
        "Model": name,
        "CV ROC-AUC (Mean)": scores.mean(),
        "CV ROC-AUC (Std)": scores.std(),
        "Test ROC-AUC": test_auc
    })

benchmark_df = pd.DataFrame(records)
print("=== Model Benchmark Summary ===")
benchmark_df

In [ ]:
# 6. Hyperparameter Optimization with Optuna (Random Forest)
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 250),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "random_state": 42
    }
    rf = RandomForestClassifier(**params)
    score = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean()
    return score

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
print("Optuna Best Parameters:", study.best_params)
print(f"Optuna Best CV ROC-AUC: {study.best_value:.4f}")

In [ ]:
# 7. Deep Learning Benchmark: Multi-Layer Perceptron (MLP)
def build_mlp(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])
    return model

mlp = build_mlp(X_train_scaled.shape[1])
es = callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
history = mlp.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    callbacks=[es],
    verbose=0
)
mlp_preds = mlp.predict(X_test_scaled).flatten()
print(f"Keras MLP Test ROC-AUC: {roc_auc_score(y_test, mlp_preds):.4f}")

In [ ]:
# 8. Comprehensive Model Diagnostics & ROC Curves
best_rf = RandomForestClassifier(**study.best_params, random_state=42)
best_rf.fit(X_train, y_train)
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

print("=== Final Classification Report (Optimized Random Forest) ===")
print(classification_report(y_test, y_pred_rf, target_names=["No Disease", "Heart Disease"]))

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_preds)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, label=f"Optimized Random Forest (AUC = {roc_auc_score(y_test, y_proba_rf):.3f})", lw=2)
plt.plot(fpr_mlp, tpr_mlp, label=f"Keras Neural Network MLP (AUC = {roc_auc_score(y_test, mlp_preds):.3f})", lw=2)
plt.plot([0, 1], [0, 1], "k--", label="Random Chance Baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Comparative ROC Curves for Heart Disease Prediction")
plt.legend(loc="lower right")
plt.show()